# Dal laboratorio al dispositivo

*Il team di WristMind ha un modello che funziona. Ma il braccialetto non gira su Google Colab: gira su un chip piccolo come un'unghia, con pochi kilobyte di RAM e qualche milliwatt di potenza. Come trasformiamo il nostro modello Keras in qualcosa che possa vivere su quel chip?*

**In questo notebook vedremo tutto il percorso:**
1. Addestriamo il modello finale (compatto e robusto)
2. Lo convertiamo in formato TFLite (TensorFlow Lite)
3. Lo quantizziamo: riduzione del 75% delle dimensioni senza perdere accuratezza
4. Scopriamo come funziona l'inferenza su microcontrollore
5. Confrontiamo le dimensioni e le prestazioni delle tre versioni

In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1" 

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

import tensorflow as tf
from tensorflow import keras

np.random.seed(42)
tf.random.set_seed(42)

print("Carico il dataset HAR...")
har = fetch_openml(data_id=1478, as_frame=True, parser='auto')
X = har.data.astype(np.float32)
y = har.target.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train).astype(np.float32)
X_test_sc  = scaler.transform(X_test).astype(np.float32)

y_train_0 = (y_train - 1).values
y_test_0  = (y_test  - 1).values

print(f"Training: {X_train_sc.shape[0]} campioni, Test: {X_test_sc.shape[0]} campioni")

ImportError: cannot import name 'keras' from 'tensorflow' (unknown location)

## Addestriamo il modello finale

Per il deployment scegliamo un'architettura **bilanciata**: abbastanza profonda da essere accurata, abbastanza piccola da stare nel chip. Aggiungiamo Dropout leggero (0.2) per robustezza.

In [ ]:
# Modello finale WristMind: bilanciato tra prestazioni e dimensioni
modello_finale = keras.Sequential([
    keras.layers.Dense(128, activation='relu', input_shape=(X_train_sc.shape[1],)),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(6, activation='softmax')
], name="wristmind_finale")

modello_finale.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(f"Parametri totali: {modello_finale.count_params():,}")
print(f"Dimensione teorica (float32, 4 byte/param): "
      f"{modello_finale.count_params() * 4 / 1024:.1f} KB")
print()

storia = modello_finale.fit(
    X_train_sc, y_train_0,
    epochs=60, batch_size=64, validation_split=0.2, verbose=0
)

_, acc_finale = modello_finale.evaluate(X_test_sc, y_test_0, verbose=0)
print(f"Accuratezza modello finale: {acc_finale:.1%}")

ImportError: Keras cannot be imported. Check that it is installed.

In [ ]:
# Salviamo il modello in formato Keras nativo
CARTELLA = "/content" if os.path.exists("/content") else "/tmp"
PATH_KERAS = f"{CARTELLA}/modello_wristmind.keras"

modello_finale.save(PATH_KERAS)
size_keras_kb = os.path.getsize(PATH_KERAS) / 1024
print(f"Modello Keras salvato: {PATH_KERAS}")
print(f"Dimensione su disco: {size_keras_kb:.1f} KB")

Modello Keras salvato: /tmp/modello_wristmind.keras
Dimensione su disco: 973.5 KB


## TensorFlow Lite: il formato per i dispositivi

Il formato `.keras` non e' progettato per microcontrollori: contiene metadati, strutture dati Python e molto overhead. **TensorFlow Lite** (TFLite) è il formato ottimizzato per dispositivi embedded:

- Struttura dati compatta (FlatBuffer, non JSON)
- Operazioni fusion: più operazioni combinate in una sola
- Nessuna dipendenza da Python o da TensorFlow completo
- Supporto su ARM Cortex-M, ESP32, Arduino Nano 33 BLE, Raspberry Pi

La conversione da Keras a TFLite richiede una sola chiamata API:

In [ ]:
# Versione 1: conversione base (pesi in float32, nessuna ottimizzazione)
converter_f32 = tf.lite.TFLiteConverter.from_keras_model(modello_finale)

tflite_f32 = converter_f32.convert()

PATH_TFLITE_F32 = f"{CARTELLA}/wristmind_f32.tflite"
with open(PATH_TFLITE_F32, 'wb') as f:
    f.write(tflite_f32)

size_f32_kb = os.path.getsize(PATH_TFLITE_F32) / 1024
print(f"TFLite float32: {size_f32_kb:.1f} KB")
print(f"Riduzione rispetto a Keras: {(1 - size_f32_kb/size_keras_kb)*100:.0f}%")

TypeError: 'NoneType' object is not callable

## Quantizzazione: da 4 byte a 1 byte per peso

Il trucco più potente per ridurre le dimensioni è la **quantizzazione int8**: i pesi del modello sono rappresentati come interi a 8 bit invece di float a 32 bit. Il risparmio è esattamente **4x**.

Come funziona? Il convertitore TFLite analizza un campione di dati reali (*representative dataset*) per capire il range tipico dei valori di attivazione, e mappa quei valori nel range [-128, 127] degli interi a 8 bit.

La precisione numerica si riduce leggermente (8 bit invece di 32), ma nella pratica l'accuratezza del modello cala pochissimo — spesso meno dell'1%.

In [ ]:
# Versione 2: quantizzazione int8 (pesi da float32 a int8)
converter_int8 = tf.lite.TFLiteConverter.from_keras_model(modello_finale)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]

# Il "representative dataset" aiuta il convertitore a calibrare la quantizzazione
# Si usa un campione del training set (non serve tutto)
def representative_dataset():
    per_batch = 100  # 100 campioni per batch di calibrazione
    for i in range(0, min(1000, len(X_train_sc)), per_batch):
        batch = X_train_sc[i:i+per_batch]
        yield [batch]

converter_int8.representative_dataset = representative_dataset
# Forziamo int8 anche per input e output (massima ottimizzazione)
converter_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_int8.inference_input_type  = tf.float32  # input resta float per semplicita'
converter_int8.inference_output_type = tf.float32

tflite_int8 = converter_int8.convert()

PATH_TFLITE_INT8 = f"{CARTELLA}/wristmind_int8.tflite"
with open(PATH_TFLITE_INT8, 'wb') as f:
    f.write(tflite_int8)

size_int8_kb = os.path.getsize(PATH_TFLITE_INT8) / 1024
print(f"TFLite int8 quantizzato: {size_int8_kb:.1f} KB")
print(f"Riduzione rispetto a float32: {(1 - size_int8_kb/size_f32_kb)*100:.0f}%")
print(f"Riduzione rispetto a Keras:   {(1 - size_int8_kb/size_keras_kb)*100:.0f}%")

## Inferenza TFLite: come funziona il riconoscimento sul chip

Per fare una previsione con il modello TFLite, non si usa il normale `model.predict()`. Si crea un **Interpreter**: un motore di esecuzione leggero che carica il modello, alloca i buffer di memoria, e gestisce input/output.

Questo è esattamente il codice che gira (con le opportune librerie C) su un microcontrollore:

In [ ]:
def inferenza_tflite(modello_bytes, X):
    """Esegue inferenza con un modello TFLite su un batch di dati."""
    # Crea l'interprete e alloca i tensori
    interp = tf.lite.Interpreter(model_content=modello_bytes)
    interp.allocate_tensors()

    # Recupera informazioni su input e output
    info_input  = interp.get_input_details()[0]
    info_output = interp.get_output_details()[0]

    predizioni = []
    for campione in X:
        # Prepara il tensore di input: shape (1, 561) con tipo float32
        tensore_in = campione.reshape(1, -1).astype(np.float32)
        interp.set_tensor(info_input['index'], tensore_in)

        # Esegui la forward pass
        interp.invoke()

        # Leggi l'output: vettore di 6 probabilita'
        output = interp.get_tensor(info_output['index'])
        predizioni.append(np.argmax(output))

    return np.array(predizioni)

# Valutiamo l'accuratezza dei due modelli TFLite sul test set
print("Eseguo inferenza TFLite float32...")
y_pred_f32  = inferenza_tflite(tflite_f32,  X_test_sc)
acc_f32 = accuracy_score(y_test_0, y_pred_f32)
print(f"Accuratezza TFLite float32: {acc_f32:.1%}")

print("Eseguo inferenza TFLite int8...")
y_pred_int8 = inferenza_tflite(tflite_int8, X_test_sc)
acc_int8 = accuracy_score(y_test_0, y_pred_int8)
print(f"Accuratezza TFLite int8:    {acc_int8:.1%}")

print(f"\nCalo di accuratezza dalla quantizzazione: {(acc_f32 - acc_int8)*100:.2f} punti percentuali")

In [ ]:
# Confronto visivo: dimensioni e accuratezza delle tre versioni del modello
modelli_nomi   = ["Keras (.keras)", "TFLite float32", "TFLite int8"]
modelli_size   = [size_keras_kb, size_f32_kb, size_int8_kb]
modelli_acc    = [acc_finale * 100, acc_f32 * 100, acc_int8 * 100]
colori         = ['#264653', '#2a9d8f', '#e76f51']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Dimensioni
bars1 = ax1.bar(modelli_nomi, modelli_size, color=colori, width=0.5, edgecolor='white')
ax1.set_ylabel('Kilobyte')
ax1.set_title('Dimensione del modello')
ax1.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars1, modelli_size):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             f'{val:.0f} KB', ha='center', fontsize=10, fontweight='bold')

# Accuratezza
bars2 = ax2.bar(modelli_nomi, modelli_acc, color=colori, width=0.5, edgecolor='white')
ax2.set_ylabel('Accuratezza (%)')
ax2.set_title('Accuratezza sul test set')
ax2.set_ylim([85, 100])
ax2.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars2, modelli_acc):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.1,
             f'{val:.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Confronto versioni del modello WristMind',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Dal file al chip: l'array C

Su un microcontrollore non esiste il filesystem come lo conoscete su un PC. I programmi vengono compilati in un unico file binario che include sia il codice che i dati. Per includere il modello TFLite nel firmware, lo convertiamo in un **array C**: una variabile che contiene i byte del modello.

Ecco come appare l'inizio del file `wristmind_model.h` che includereste nel progetto Arduino:

```c
// wristmind_model.h - generato automaticamente
// Dimensione: ~XX KB - TFLite int8 quantizzato

#ifndef WRISTMIND_MODEL_H
#define WRISTMIND_MODEL_H

const unsigned char wristmind_model_data[] = {
    0x1c, 0x00, 0x00, 0x00, 0x54, 0x46, 0x4c, 0x33,
    0x14, 0x00, 0x20, 0x00, ...
};
const unsigned int wristmind_model_data_len = XXXX;

#endif
```

Questo header file viene poi incluso nel codice Arduino e passato all'interprete TFLite for Microcontrollers.

In [ ]:
# Generiamo i primi 32 byte dell'array C come anteprima
byte_modello = tflite_int8
print(f"// File: wristmind_model.h")
print(f"// Dimensione totale: {len(byte_modello)} byte ({len(byte_modello)/1024:.1f} KB)")
print(f"")
print(f"const unsigned char wristmind_model_data[] = {{")
print(f"    ", end="")
for i, b in enumerate(byte_modello[:32]):
    if i > 0 and i % 8 == 0:
        print()
        print(f"    ", end="")
    print(f"0x{b:02x}", end=", " if i < 31 else "")
print("  // ... continua")
print(f"}};")
print(f"const unsigned int wristmind_model_data_len = {len(byte_modello)};")

## Il firmware del braccialetto

Il codice che gira sul microcontrollore (ad esempio Arduino Nano 33 BLE Sense) segue sempre lo stesso schema:

```cpp
// wristmind_inference.ino
#include <TensorFlowLite.h>
#include "wristmind_model.h"

// Allocazione del tensore: area di memoria riservata al runtime TFLite
// Deve essere abbastanza grande da contenere tutti i tensori intermedi
constexpr int kTensorArenaSize = 20 * 1024;  // 20 KB
uint8_t tensor_arena[kTensorArenaSize];

void setup() {
    // 1. Carica il modello dall'array C in memoria flash
    const tflite::Model* model = tflite::GetModel(wristmind_model_data);

    // 2. Crea l'interprete con i resolver delle operazioni usate
    static tflite::MicroMutableOpResolver<4> resolver;
    resolver.AddFullyConnected();   // Dense layers
    resolver.AddSoftmax();          // Attivazione output
    resolver.AddRelu();             // Attivazione strati nascosti
    resolver.AddDequantize();       // Conversione int8 -> float

    // 3. Crea e inizializza l'interprete
    static tflite::MicroInterpreter interpreter(
        model, resolver, tensor_arena, kTensorArenaSize);
    interpreter.AllocateTensors();

    // 4. Punta ai buffer di input e output
    TfLiteTensor* input  = interpreter.input(0);
    TfLiteTensor* output = interpreter.output(0);
}

void loop() {
    // 5. Leggi i sensori (accelerometro + giroscopio)
    float raw_data[561];
    leggi_e_preprocessa_sensori(raw_data);  // normalizzazione con media/std

    // 6. Copia nel tensore di input e lancia l'inferenza
    for (int i = 0; i < 561; i++) input->data.f[i] = raw_data[i];
    interpreter.Invoke();

    // 7. Leggi l'output: indice della probabilita' massima = attivita'
    int classe = 0;
    float prob_max = 0;
    for (int k = 0; k < 6; k++) {
        if (output->data.f[k] > prob_max) {
            prob_max = output->data.f[k]; classe = k;
        }
    }

    const char* attivita[] = {"Cammina","Salisce","Scende","Seduto","In piedi","Sdraiato"};
    Serial.println(attivita[classe]);
    delay(200);  // 5 inferenze al secondo
}
```

Questo è il codice reale che gira su chip come l'Arduino Nano 33 BLE Sense o l'ESP32. La TensorFlow Lite for Microcontrollers library — open source — traduce le operazioni della rete in istruzioni ottimizzate per il processore ARM.

## Panoramica dell'hardware Edge AI

Quale hardware scegliere per il braccialetto WristMind? Dipende dal trade-off tra prestazioni, dimensioni e consumo energetico:

In [ ]:
# Tabella comparativa hardware per edge AI
df_hw = pd.DataFrame({
    "Dispositivo": [
        "Laptop (GPU/CPU)",
        "Raspberry Pi 4",
        "Arduino Nano 33 BLE",
        "ESP32",
        "Nordic nRF52840"
    ],
    "CPU / MCU": [
        "x86/ARM, GHz",
        "ARM Cortex-A72, 1.5 GHz",
        "ARM Cortex-M4, 64 MHz",
        "Xtensa LX6, 240 MHz",
        "ARM Cortex-M4, 64 MHz"
    ],
    "RAM": ["8-32 GB", "4 GB", "256 KB", "520 KB", "256 KB"],
    "Flash": ["SSD", "SD card", "1 MB", "4 MB", "1 MB"],
    "Consumo": ["15-150 W", "3-5 W", "~10 mW", "~80 mW", "~4 mW"],
    "TFLite": [
        "TF completo",
        "TFLite full",
        "TFLite Micro",
        "TFLite Micro",
        "TFLite Micro"
    ],
    "Caso d'uso": [
        "Training + sviluppo",
        "Edge server, IoT gateway",
        "Wearable, sensori",
        "IoT, prototipazione",
        "Wearable premium"
    ]
})
print(df_hw.to_string(index=False))

## Il valore dell'Edge AI

Perché fare l'inferenza sul chip invece di inviare i dati a un server cloud?

**Privacy**: i dati grezzi dei sensori rimangono sul dispositivo. Nessun dato personale (come le tue attività quotidiane) viene trasmesso in rete.

**Velocità**: l'inferenza sul chip avviene in pochi millisecondi. Non c'è latenza di rete, nessun ritardo dovuto alla connessione.

**Autonomia**: il dispositivo funziona anche senza connessione internet. Un braccialetto che smette di funzionare in palestra perché non c'è WiFi non è un buon prodotto.

**Energia**: trasmettere dati via Bluetooth o WiFi consuma molto più energia del fare un calcolo sul microcontrollore. L'Edge AI può prolungare la batteria di ore.

In [ ]:
# Visualizziamo la pipeline completa del sistema WristMind
fig, asse = plt.subplots(figsize=(14, 4))
asse.set_xlim(0, 14)
asse.set_ylim(0, 4)
asse.axis('off')

import matplotlib.patches as mpatches

# Definiamo le fasi della pipeline
fasi = [
    (0.3,  "SENSORI\nAccelero.\n+ Giroscopio", "#264653"),
    (2.5,  "FINESTRA\n2 secondi\n561 feature",  "#457b9d"),
    (4.7,  "NORM.\nStandard\nScaler",           "#2a9d8f"),
    (6.9,  "MLP\nTFLite\nint8",                 "#e76f51"),
    (9.1,  "RISULTATO\n6 classi\n+ proba.",      "#e9c46a"),
    (11.3, "AZIONE\nDisplay\n+ Alert",           "#f4a261"),
]

for x, testo, colore in fasi:
    rect = mpatches.FancyBboxPatch((x, 0.8), 1.8, 2.4,
                                    boxstyle="round,pad=0.1",
                                    facecolor=colore, edgecolor='white',
                                    linewidth=1.5, alpha=0.9)
    asse.add_patch(rect)
    asse.text(x + 0.9, 2.0, testo, ha='center', va='center',
              fontsize=8.5, color='white', fontweight='bold',
              multialignment='center')
    if x < 11.3:
        asse.annotate("", xy=(x + 2.1, 2.0), xytext=(x + 1.8, 2.0),
                      arrowprops=dict(arrowstyle="->", color="#555555", lw=1.5))

asse.text(7.0, 0.3, "Tutto gira sul microcontrollore — nessun cloud necessario",
          ha='center', fontsize=9.5, color='#555555', fontstyle='italic')

plt.title("Pipeline di inferenza WristMind sul chip", fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.show()

## Tabella riassuntiva: le tre versioni del modello

In [ ]:
# Tabella finale comparativa delle tre versioni
df_riepilogo = pd.DataFrame({
    "Versione": [
        "Keras (.keras)",
        "TFLite float32",
        "TFLite int8 quantizzato"
    ],
    "Formato": ["HDF5 + JSON", "FlatBuffer", "FlatBuffer"],
    "Precisione pesi": ["float32 (4 B)", "float32 (4 B)", "int8 (1 B)"],
    "Dimensione (KB)": [
        f"{size_keras_kb:.0f}",
        f"{size_f32_kb:.0f}",
        f"{size_int8_kb:.0f}"
    ],
    "Accuratezza test": [
        f"{acc_finale:.1%}",
        f"{acc_f32:.1%}",
        f"{acc_int8:.1%}"
    ],
    "Destinazione": [
        "Solo sviluppo",
        "Raspberry Pi, telefono",
        "Arduino, ESP32, MCU"
    ]
})
print(df_riepilogo.to_string(index=False))

print()
print(f"Riduzione totale (Keras -> int8): {(1 - size_int8_kb/size_keras_kb)*100:.0f}%")
print(f"Calo accuratezza: {(acc_finale - acc_int8)*100:.2f} punti percentuali")

---

> **Cosa abbiamo imparato**: Il percorso da un modello Keras a un chip embedded passa attraverso TensorFlow Lite. La conversione preserva la struttura della rete ottimizzandone il formato; la quantizzazione int8 riduce le dimensioni di 4x a costo di un calo minimo di accuratezza. L'inferenza su microcontrollore usa un Interpreter leggero che non dipende da Python né da TensorFlow completo. I modelli TFLite vengono incorporati nel firmware come array C e gira con pochi kilobyte di RAM.

> **Benvenuti nell'ingegneria dell'informazione**: In quattro notebook siete passati da un singolo neurone artificiale a un sistema di edge AI funzionante: raccolta dati dai sensori, preprocessing, addestramento di una rete neurale profonda, valutazione delle prestazioni, ottimizzazione e deployment su chip embedded. Questa è la catena completa del machine learning applicato. I principi che avete visto  (discesa del gradiente, backpropagation, overfitting, regolarizzazione, quantizzazione) sono quelli usati ogni giorno nei laboratori delle aziende che costruiscono i dispositivi che usate. La fisica, la matematica e il codice si incontrano qui: nell'ingegneria dell'informazione.